# 🚀 Qwen Image Editor Studio - Kaggle GPU Backend (Tesla T4 16GB Edition)
This notebook runs the **FastAPI Image Editor Backend Engine** with 4-step Qwen-Image-Edit, Civitai LoRA gallery sync, Google Drive Vault, dynamic idle timeout auto-shutdown, and Cloudflare Ingress.

In [ ]:
# ─── CELL 1: DEPENDENCY INSTALLATION ──────────────────────────────────
!apt-get update -qq && apt-get install -y -qq aria2 curl > /dev/null 2>&1
!curl -fsSL https://rclone.org/install.sh | bash > /dev/null 2>&1 || apt-get install -y -qq rclone > /dev/null 2>&1
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb && dpkg -i cloudflared.deb > /dev/null 2>&1 && rm cloudflared.deb
!pip install -q --no-cache-dir git+https://github.com/huggingface/diffusers.git transformers accelerate peft sentencepiece protobuf uvicorn fastapi pillow requests google-api-python-client google-auth-httplib2 google-auth-oauthlib pydantic bitsandbytes python-multipart
!echo "✅ System Check:" && rclone --version | head -n 1 && cloudflared --version


In [ ]:
# ─── CELL 2: RUNTIME BACKEND ENGINE ─────────────────────────────────
import base64, pathlib
b64_code = """..."""

server_py = pathlib.Path("/kaggle/working/server.py")
server_py.write_text(base64.b64decode(b64_code).decode("utf-8"))
print("✅ /kaggle/working/server.py written successfully.")

!python /kaggle/working/server.py


In [ ]:
# ─── CELL 3: BOOT BACKEND & CLOUDFLARE INGRESS ────────────────────────
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True && export CUDA_MODULE_LOADING=LAZY && python /kaggle/working/server.py
